# Modes of action (OCD) with Isoacute Comparison

In [1]:
!pip install --upgrade sympy
# Then restart runtime (Runtime -> Restart runtime)

In [2]:
#!/usr/bin/env python3
"""
================================================================================
COMPUTATIONAL MODEL OF OCD-LIKE RIGIDITY — v4.0 BIOLOGICALLY-GROUNDED REVISION
================================================================================
This revision responds directly to Reviewer 1 & 2 critiques (28 Feb 2026).
================================================================================
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from typing import List, Tuple, Optional, Dict, Any
from dataclasses import dataclass
from enum import Enum
import copy
import warnings

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
CONFIG = {
    # Architecture
    'input_dim': 2, 'hidden_dims': [128, 64], 'output_dim': 4, 'num_gru_layers': 2,
    'rnn_type': 'gru',          # 'gru' | 'lstm'  (ablation)
    'use_modular': True,        # striatal+thalamic modules on/off (ablation)
    'striatal_recurrence_bias': 1.5,  # habit module pruned-protection

    # Training
    'batch_size': 32, 'baseline_lr': 1e-3, 'finetune_lr': 5e-4,
    'baseline_epochs': 40, 'regrowth_epochs': 30,

    # Task
    'seq_len': 200, 'n_train_sequences': 400, 'n_test_sequences': 100,
    'switch_interval': 50, 'n_rules': 4,
    'compulsive_task': False,   # optional sticky "ritual" actions (conceptual)
    'ritual_prob': 0.08, 'ritual_action': 0,

    # Pruning
    'regrowth_fraction': 0.50, 'regrowth_init_scale': 0.03, 'recurrence_bias': 1.2,
    # CHANGED: biologically motivated default + sensitivity range
    'ocd_prune_sparsity': 0.60,
    'prune_mode': 'activity',   # 'activity' (complement-like) | 'magnitude'
    'prune_sensitivity_range': [0.40, 0.50, 0.60, 0.70],
    'activity_n_batches': 20,

    # Relapse (now cumulative + stress-dependent; labelled conceptual)
    'relapse_prune_fraction': 0.40,
    'relapse_mode': 'cumulative',   # 'single' | 'cumulative'
    'relapse_weeks': 4, 'relapse_per_week': 0.12, 'relapse_stress_gain': 0.5,
    'relapse_noise': 0.2,

    # Treatments
    'comparison_ketamine_regrow': 0.60, 'comparison_ketamine_epochs': 10,
    'comparison_ssri_epochs': 120, 'comparison_ssri_lr': 1e-5,
    'comparison_ssri_initial_stress': 0.4,
    'comparison_neurosteroid_strength': 0.65,
    'comparison_neurosteroid_use_tanh': True, 'comparison_neurosteroid_epochs': 8,

    # Iso-dose
    'iso_dose_norm_type': 'l1',
    'iso_dose_target_doses': [0.005, 0.010, 0.020, 0.040],
    'iso_dose_tolerance': 0.002, 'iso_dose_turnover_threshold': 0.10,
    'ketamine_regrow_sweep': [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80],
    'ssri_epochs_sweep': [20, 40, 60, 80, 100, 120, 160, 200],
    'neurosteroid_strength_sweep': [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85],

    # Reproducibility & statistics
    'seed': 42, 'n_seeds': 5,
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed: int):
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def print_section_header(title, width=80, char="="):
    print(f"\n{char*width}\n{title.center(width)}\n{char*width}")


def print_subsection_header(title, width=60, char="-"):
    print(f"\n{char*width}\n  {title}\n{char*width}")


# =============================================================================
# DOSING METRICS (multi-metric; documented as computational convenience)
# =============================================================================
def compute_weight_change_norm(pre_state, model_post, norm_type='l1'):
    delta, total = 0.0, 0
    for name, p in model_post.named_parameters():
        if 'weight' in name and name in pre_state:
            diff = (p.data - pre_state[name]).abs()
            total += p.numel()
            delta += diff.sum().item() if norm_type == 'l1' else (diff**2).sum().item()
    if norm_type == 'l2':
        delta = delta ** 0.5
    return delta / total if total > 0 else 0.0


def compute_synaptic_turnover(pre_state, model_post, threshold=0.10):
    changed, total = 0, 0
    for name, p in model_post.named_parameters():
        if 'weight' in name and name in pre_state:
            pre = pre_state[name]
            rel = (p.data - pre).abs() / (pre.abs().clamp(min=1e-8))
            changed += (rel > threshold).sum().item(); total += p.numel()
    return changed / total if total > 0 else 0.0


def compute_sparsity_change(pre_state, model_post):
    def sp(sd):
        t = z = 0
        for n, x in sd.items():
            if 'weight' in n:
                t += x.numel(); z += (x.abs() < 1e-8).sum().item()
        return z / t if t > 0 else 0.0
    return abs(sp({n: p.data for n, p in model_post.named_parameters()}) - sp(pre_state))


@dataclass
class DoseMetrics:
    l1_norm: float = 0.0; l2_norm: float = 0.0
    synaptic_turnover: float = 0.0; sparsity_change: float = 0.0
    @property
    def primary_dose(self):
        return self.l1_norm


def compute_all_dose_metrics(pre_state, model_post, turnover_threshold=None):
    if turnover_threshold is None:
        turnover_threshold = CONFIG['iso_dose_turnover_threshold']
    return DoseMetrics(
        l1_norm=compute_weight_change_norm(pre_state, model_post, 'l1'),
        l2_norm=compute_weight_change_norm(pre_state, model_post, 'l2'),
        synaptic_turnover=compute_synaptic_turnover(pre_state, model_post, turnover_threshold),
        sparsity_change=compute_sparsity_change(pre_state, model_post))


def composite_dose(dm: DoseMetrics, scales: Dict[str, float]) -> float:
    """Min-max-scaled composite across L1/L2/turnover/Δsparsity (computational only)."""
    def nz(x, lo, hi):
        return 0.0 if hi - lo < 1e-12 else (x - lo) / (hi - lo)
    return float(np.mean([
        nz(dm.l1_norm, *scales['l1']), nz(dm.l2_norm, *scales['l2']),
        nz(dm.synaptic_turnover, *scales['turnover']),
        nz(dm.sparsity_change, *scales['sparsity'])]))


# =============================================================================
# TASK DEFINITION
# =============================================================================
class Rule(Enum):
    X_SIGN = 0; Y_SIGN = 1; QUADRANT = 2; DIAGONAL = 3


def apply_rule(points, rule):
    x, y = points[..., 0], points[..., 1]
    if rule == Rule.X_SIGN.value:
        labels = ((x >= 0).long() * 2 + (y >= 0).long())
    elif rule == Rule.Y_SIGN.value:
        labels = ((y >= 0).long() * 2 + (x >= 0).long())
    elif rule == Rule.QUADRANT.value:
        labels = ((x >= 0).long() + (y >= 0).long() * 2)
    elif rule == Rule.DIAGONAL.value:
        labels = (y >= x).long() * 2 + (y >= -x).long()
    else:
        raise ValueError(f"Unknown rule: {rule}")
    return labels


def generate_base_points(n, noise=0.8):
    centers = torch.tensor([[1.5, 1.5], [-1.5, 1.5], [-1.5, -1.5], [1.5, -1.5]],
                           dtype=torch.float32)
    idx = torch.randint(0, 4, (n,))
    return centers[idx] + torch.randn(n, 2) * noise


def generate_rule_switch_sequences(n_sequences, seq_len, switch_interval,
                                   noise=0.8, deterministic_switches=False,
                                   compulsive=False):
    all_data, all_labels, all_rules = [], [], []
    for _ in range(n_sequences):
        points = generate_base_points(seq_len, noise=noise)
        rules = torch.zeros(seq_len, dtype=torch.long)
        cur = torch.randint(0, CONFIG['n_rules'], (1,)).item()
        if deterministic_switches:
            sw = set(range(switch_interval, seq_len, switch_interval))
        else:
            n_sw = max(1, seq_len // switch_interval)
            vr = list(range(20, seq_len - 20))
            sw = set(np.random.choice(vr, size=n_sw, replace=False)) if len(vr) >= n_sw else set(vr)
        for t in range(seq_len):
            if t in sw:
                cur = (cur + np.random.randint(1, CONFIG['n_rules'])) % CONFIG['n_rules']
            rules[t] = cur
        labels = torch.zeros(seq_len, dtype=torch.long)
        for t in range(seq_len):
            labels[t] = apply_rule(points[t:t+1], rules[t].item())[0]
        # CONCEPTUAL: occasional sticky "ritual" actions (compulsive habit loop)
        if compulsive:
            t = 0
            while t < seq_len:
                if np.random.rand() < CONFIG['ritual_prob']:
                    run = min(np.random.randint(3, 8), seq_len - t)
                    labels[t:t+run] = CONFIG['ritual_action']
                    t += run
                else:
                    t += 1
        all_data.append(points); all_labels.append(labels); all_rules.append(rules)
    return torch.stack(all_data), torch.stack(all_labels), torch.stack(all_rules)


def create_rule_switch_dataloaders(n_train=None, n_test=None, seq_len=None,
                                   batch_size=None, switch_interval=None, noise=0.8):
    n_train = n_train or CONFIG['n_train_sequences']
    n_test = n_test or CONFIG['n_test_sequences']
    seq_len = seq_len or CONFIG['seq_len']
    batch_size = batch_size or CONFIG['batch_size']
    switch_interval = switch_interval or CONFIG['switch_interval']
    comp = CONFIG['compulsive_task']
    tr_d, tr_l, tr_r = generate_rule_switch_sequences(
        n_train, seq_len, switch_interval, noise=noise,
        deterministic_switches=False, compulsive=comp)
    te_d, te_l, te_r = generate_rule_switch_sequences(
        n_test, seq_len, switch_interval, noise=noise,
        deterministic_switches=True, compulsive=comp)
    tr = DataLoader(TensorDataset(tr_d, tr_l, tr_r), batch_size=batch_size, shuffle=True)
    te = DataLoader(TensorDataset(te_d, te_l, te_r), batch_size=batch_size, shuffle=False)
    return tr, te, te_r


# =============================================================================
# ENHANCED CSTC NETWORK (modular: cortex -> striatum(habit) -> thalamic gating)
# =============================================================================
class CSTCNetwork(nn.Module):
    def __init__(self, hidden_dims=None, num_layers=None,
                 rnn_type=None, use_modular=None):
        super().__init__()
        hidden_dims = hidden_dims or CONFIG['hidden_dims']
        num_layers = num_layers or CONFIG['num_gru_layers']
        self.rnn_type = (rnn_type or CONFIG['rnn_type']).lower()
        self.use_modular = CONFIG['use_modular'] if use_modular is None else use_modular

        self.hidden_dim = hidden_dims[1]; self.num_layers = num_layers
        self.input_fc = nn.Linear(CONFIG['input_dim'], hidden_dims[0])

        rnn_cls = nn.LSTM if self.rnn_type == 'lstm' else nn.GRU
        self.rnn = rnn_cls(input_size=hidden_dims[0], hidden_size=hidden_dims[1],
                           num_layers=num_layers, batch_first=True,
                           dropout=0.1 if num_layers > 1 else 0.0)

        # Striatal "habit" module + thalamic confidence gate
        self.striatal_fc = nn.Linear(hidden_dims[1], hidden_dims[1])
        self.thalamic_gate = nn.Linear(hidden_dims[1], 1)
        self.output_fc = nn.Linear(hidden_dims[1], CONFIG['output_dim'])
        self.relu = nn.ReLU()

        self.stress_level = 0.0; self.glutamate_noise = 0.0
        self.inhibition_strength = 1.0; self.use_tanh = False

        self.register_buffer('input_mask', torch.ones_like(self.input_fc.weight))
        self.register_buffer('output_mask', torch.ones_like(self.output_fc.weight))
        self.register_buffer('striatal_mask', torch.ones_like(self.striatal_fc.weight))
        self.register_buffer('thalamic_mask', torch.ones_like(self.thalamic_gate.weight))

    # name->buffer for masked linears used in forward
    BUFFER_MAP = {'input_fc.weight': 'input_mask', 'output_fc.weight': 'output_mask',
                  'striatal_fc.weight': 'striatal_mask',
                  'thalamic_gate.weight': 'thalamic_mask'}

    def init_hidden(self, batch_size, device):
        h = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)
        if self.rnn_type == 'lstm':
            return (h, torch.zeros_like(h))
        return h

    def set_inhibition(self, strength, use_tanh=False):
        self.inhibition_strength = max(0.0, min(1.0, strength)); self.use_tanh = use_tanh

    def reduce_stress_gradually(self, epoch, total_epochs, initial_stress=0.4, final_stress=0.0):
        prog = epoch / max(total_epochs - 1, 1)
        self.stress_level = initial_stress + prog * (final_stress - initial_stress)

    def forward(self, x, hidden=None, return_hidden=False):
        single = x.dim() == 2
        if single:
            x = x.unsqueeze(1)
        bsz, slen, _ = x.shape; device = x.device
        if hidden is None:
            hidden = self.init_hidden(bsz, device)
        if self.glutamate_noise > 0:
            x = x + torch.randn_like(x) * self.glutamate_noise

        h = self.relu(F.linear(x, self.input_fc.weight * self.input_mask, self.input_fc.bias))
        h = h * self.inhibition_strength
        if self.stress_level > 0:
            h = h + torch.randn_like(h) * self.stress_level

        rnn_out, hidden = self.rnn(h, hidden)
        rnn_out = rnn_out * self.inhibition_strength
        if self.use_tanh:
            rnn_out = torch.tanh(rnn_out)
        if self.stress_level > 0:
            rnn_out = rnn_out + torch.randn_like(rnn_out) * self.stress_level * 0.5

        if self.use_modular:
            habit = self.relu(F.linear(rnn_out, self.striatal_fc.weight * self.striatal_mask,
                                       self.striatal_fc.bias))
            gate = torch.sigmoid(F.linear(rnn_out, self.thalamic_gate.weight * self.thalamic_mask,
                                          self.thalamic_gate.bias))  # (B,T,1)
            combined = gate * habit + (1.0 - gate) * rnn_out
        else:
            combined = rnn_out

        logits = F.linear(combined, self.output_fc.weight * self.output_mask, self.output_fc.bias)
        if single:
            logits = logits.squeeze(1)
        return (logits, hidden) if return_hidden else (logits, None)

    def set_stress(self, level):
        self.stress_level = max(0.0, level)

    def set_glutamate_noise(self, level):
        self.glutamate_noise = max(0.0, level)

    def get_sparsity(self):
        t = z = 0
        for n, p in self.named_parameters():
            if 'weight' in n:
                t += p.numel(); z += (p.abs() < 1e-8).sum().item()
        return z / t if t > 0 else 0.0

    def get_treatment_state(self):
        return {'stress_level': self.stress_level, 'glutamate_noise': self.glutamate_noise,
                'inhibition_strength': self.inhibition_strength, 'use_tanh': self.use_tanh,
                'sparsity': self.get_sparsity()}


def _is_recurrent_name(name):
    return ('rnn' in name) or ('striatal' in name)


# =============================================================================
# PRUNING MANAGER (adds activity-dependent pruning + cumulative relapse)
# =============================================================================
class CSTCPruningManager:
    def __init__(self, model: CSTCNetwork):
        self.model = model; self.original_weights = {}; self.masks = {}; self.history = []
        self._save_original_weights()

    def _save_original_weights(self):
        for name, p in self.model.named_parameters():
            if 'weight' in name:
                self.original_weights[name] = p.data.clone()
                self.masks[name] = torch.ones_like(p.data)

    def get_sparsity(self):
        return self.model.get_sparsity()

    def _push_mask_to_buffer(self, name, mask):
        if name in self.model.BUFFER_MAP:
            getattr(self.model, self.model.BUFFER_MAP[name]).copy_(mask)

    def activity_dependent_prune(self, train_loader, prune_fraction=None,
                                 n_batches=None):
        """Complement-like: prune low recent-usage AND weak synapses (composite tag)."""
        if prune_fraction is None:
            prune_fraction = CONFIG['ocd_prune_sparsity']
        if n_batches is None:
            n_batches = CONFIG['activity_n_batches']
        if prune_fraction <= 0:
            return {'achieved_sparsity': 0.0, 'weights_pruned': 0}

        self.model.train()
        device = next(self.model.parameters()).device
        usage = {n: torch.zeros_like(p.data) for n, p in self.model.named_parameters()
                 if 'weight' in n}
        crit = nn.CrossEntropyLoss()
        for bi, (data, labels, _) in enumerate(train_loader):
            if bi >= n_batches:
                break
            data, labels = data.to(device), labels.to(device)
            self.model.zero_grad()
            logits, _ = self.model(data)
            loss = crit(logits.view(-1, CONFIG['output_dim']), labels.view(-1))
            loss.backward()
            for n, p in self.model.named_parameters():
                if n in usage and p.grad is not None:
                    usage[n] += p.grad.abs()  # accumulated activity-driven importance

        # Composite "weakness" score: low usage * low magnitude -> tag for removal.
        # Recurrent/habit synapses get protective bias (developmental persistence).
        total_pruned = 0; layer_stats = {}
        for name, p in self.model.named_parameters():
            if name not in usage:
                continue
            score = usage[name] * p.data.abs()
            if _is_recurrent_name(name):
                score = score * CONFIG['recurrence_bias']  # protect recurrent loops
            flat = score.flatten()
            k = int(prune_fraction * flat.numel())
            if k == 0:
                continue
            thr = torch.kthvalue(flat, k).values
            mask = (score > thr).float()
            pruned = (mask == 0).sum().item()
            self.masks[name] = mask; p.data *= mask
            self._push_mask_to_buffer(name, mask)
            total_pruned += pruned
            layer_stats[name] = {'pruned': pruned, 'total': p.numel(),
                                 'layer_sparsity': pruned / p.numel()}
        achieved = self.get_sparsity()
        self.history.append({'operation': 'activity_prune', 'target': prune_fraction,
                             'achieved_sparsity': achieved, 'weights_pruned': total_pruned})
        return {'achieved_sparsity': achieved, 'weights_pruned': total_pruned,
                'layer_stats': layer_stats}

    def prune_by_magnitude(self, sparsity, recurrence_bias=None):
        if recurrence_bias is None:
            recurrence_bias = CONFIG['recurrence_bias']
        if sparsity <= 0:
            return {'achieved_sparsity': 0.0, 'weights_pruned': 0}
        all_w, info = [], []
        for name, p in self.model.named_parameters():
            if 'weight' in name and p.requires_grad:
                fw = p.data.abs().flatten()
                if _is_recurrent_name(name) and recurrence_bias != 1.0:
                    fw = fw / recurrence_bias
                all_w.append(fw); info.append((name, p))
        if not all_w:
            return {'achieved_sparsity': 0.0, 'weights_pruned': 0}
        cat = torch.cat(all_w); k = int(sparsity * cat.numel())
        if k == 0:
            return {'achieved_sparsity': 0.0, 'weights_pruned': 0}
        thr = torch.kthvalue(cat, k).values.item()
        total_pruned = 0; layer_stats = {}
        for name, p in info:
            ew = p.data.abs()
            if _is_recurrent_name(name) and recurrence_bias != 1.0:
                ew = ew / recurrence_bias
            mask = (ew > thr).float(); pruned = (mask == 0).sum().item()
            self.masks[name] = mask; p.data *= mask
            total_pruned += pruned
            layer_stats[name] = {'pruned': pruned, 'total': p.numel(),
                                 'layer_sparsity': pruned / p.numel()}
            self._push_mask_to_buffer(name, mask)
        achieved = self.get_sparsity()
        self.history.append({'operation': 'prune_magnitude', 'target_sparsity': sparsity,
                             'achieved_sparsity': achieved, 'weights_pruned': total_pruned})
        return {'achieved_sparsity': achieved, 'weights_pruned': total_pruned,
                'layer_stats': layer_stats}

    def prune_baseline(self, train_loader, sparsity):
        """Dispatch to configured pruning mode."""
        if CONFIG['prune_mode'] == 'activity':
            return self.activity_dependent_prune(train_loader, prune_fraction=sparsity)
        return self.prune_by_magnitude(sparsity=sparsity)

    def gradient_guided_regrow(self, train_loader=None, regrow_fraction=None,
                               n_batches=5, init_scale=None):
        if regrow_fraction is None:
            regrow_fraction = CONFIG['regrowth_fraction']
        if init_scale is None:
            init_scale = CONFIG['regrowth_init_scale']
        if train_loader is None:
            train_loader, _, _ = create_rule_switch_dataloaders()
        self.model.train(); device = next(self.model.parameters()).device
        gi = {n: torch.zeros_like(p.data) for n, p in self.model.named_parameters() if 'weight' in n}
        crit = nn.CrossEntropyLoss()
        for bi, (data, labels, _) in enumerate(train_loader):
            if bi >= n_batches:
                break
            data, labels = data.to(device), labels.to(device)
            self.model.zero_grad()
            logits, _ = self.model(data)
            loss = crit(logits.view(-1, CONFIG['output_dim']), labels.view(-1))
            loss.backward()
            for n, p in self.model.named_parameters():
                if n in gi and p.grad is not None:
                    gi[n] += p.grad.abs()
        total_regrown = 0; layer_stats = {}
        for name, p in self.model.named_parameters():
            if 'weight' not in name:
                continue
            mask = self.masks.get(name, torch.ones_like(p.data))
            pruned_mask = (mask < 0.5)
            if pruned_mask.sum() == 0:
                continue
            n_regrow = int(regrow_fraction * pruned_mask.sum().item())
            if n_regrow == 0:
                continue
            imp = (gi[name] * pruned_mask.float()).flatten()
            n_pos = (imp > 0).sum().item()
            if n_pos == 0:
                continue
            _, top = torch.topk(imp, min(n_regrow, n_pos))
            fm = mask.flatten(); fp = p.data.flatten(); fo = self.original_weights[name].flatten()
            for idx in top:
                fm[idx] = 1.0; fp[idx] = fo[idx] * init_scale
            new_mask = fm.view_as(mask)
            p.data = fp.view_as(p.data); self.masks[name] = new_mask
            self._push_mask_to_buffer(name, new_mask)
            total_regrown += len(top)
            layer_stats[name] = {'regrown': len(top),
                                 'remaining_pruned': pruned_mask.sum().item() - len(top)}
        ns = self.get_sparsity()
        self.history.append({'operation': 'gradient_regrow', 'regrow_fraction': regrow_fraction,
                             'connections_regrown': total_regrown, 'new_sparsity': ns})
        return {'connections_regrown': total_regrown, 'new_sparsity': ns, 'layer_stats': layer_stats}

    def secondary_prune(self, fraction, bias_recurrent=False, recurrence_multiplier=1.5):
        stats = {}; total_pruned = 0
        for name, p in self.model.named_parameters():
            if name not in self.masks:
                continue
            mask = self.masks[name]; active = (mask == 1); na = active.sum().item()
            if na == 0:
                continue
            ef = fraction
            if bias_recurrent and _is_recurrent_name(name):
                ef = min(fraction * recurrence_multiplier, 0.9)
            ntp = int(ef * na)
            if ntp == 0:
                continue
            w = p.data.abs(); wa = w.clone(); wa[~active] = float('inf')
            thr = torch.kthvalue(wa.flatten(), ntp).values.item()
            pm = (w <= thr) & active
            mask[pm] = 0; p.data[pm] = 0
            pruned = pm.sum().item(); total_pruned += pruned
            stats[name] = {'pruned': pruned, 'remaining': na - pruned, 'effective_fraction': ef}
            self._push_mask_to_buffer(name, mask)
        ns = self.get_sparsity()
        self.history.append({'operation': 'secondary_prune', 'fraction': fraction,
                             'total_pruned': total_pruned, 'new_sparsity': ns})
        return {'total_pruned': total_pruned, 'new_sparsity': ns, 'layer_stats': stats}

    def cumulative_relapse(self, model, test_loader, device,
                           weeks=None, per_week=None, stress_gain=None):
        """CONCEPTUAL relapse: small recurrent-biased pruning per 'week' + rising stress."""
        weeks = weeks or CONFIG['relapse_weeks']
        per_week = per_week or CONFIG['relapse_per_week']
        stress_gain = CONFIG['relapse_stress_gain'] if stress_gain is None else stress_gain
        traj = []
        base_stress = model.stress_level
        for wk in range(1, weeks + 1):
            self.secondary_prune(fraction=per_week, bias_recurrent=True)
            model.set_stress(base_stress + stress_gain * (wk / weeks) * CONFIG['relapse_noise'])
            m = compute_ocd_metrics(model, test_loader, device)
            traj.append({'week': wk, 'sparsity': m.sparsity,
                         'persev': m.perseverative_error_rate, 'flex': m.flexibility_index})
        model.set_stress(base_stress)
        return traj

    def apply_masks(self):
        for name, p in self.model.named_parameters():
            if name in self.masks:
                p.data *= self.masks[name]


# =============================================================================
# OCD-RELEVANT METRICS
# =============================================================================
@dataclass
class OCDMetrics:
    accuracy: float = 0.0; perseverative_error_rate: float = 0.0; switch_cost: float = 0.0
    trials_to_recover: float = 0.0; repetition_rate: float = 0.0; repetition_entropy: float = 0.0
    output_diversity: float = 0.0; rule_inference_accuracy: float = 0.0
    flexibility_index: float = 0.0; sparsity: float = 0.0; stress_level: float = 0.0
    glutamate_noise: float = 0.0; inhibition_strength: float = 1.0; use_tanh: bool = False


def compute_ocd_metrics(model, test_loader, device, detailed=False):
    model.eval(); metrics = OCDMetrics()
    preds_all, labels_all, rules_all, correct_all = [], [], [], []
    with torch.no_grad():
        for data, labels, rules in test_loader:
            data, labels, rules = data.to(device), labels.to(device), rules.to(device)
            bsz, slen, _ = data.shape
            hidden = model.init_hidden(bsz, device)
            bp = []
            for t in range(slen):
                logits, hidden = model(data[:, t:t+1, :], hidden, return_hidden=True)
                bp.append(logits.squeeze(1).argmax(dim=-1))
            bp = torch.stack(bp, dim=1)
            preds_all.append(bp.cpu()); labels_all.append(labels.cpu())
            rules_all.append(rules.cpu()); correct_all.append((bp == labels).cpu())
    predictions = torch.cat(preds_all, 0); labels = torch.cat(labels_all, 0)
    rules = torch.cat(rules_all, 0); correct = torch.cat(correct_all, 0)
    n_seq, slen = predictions.shape
    metrics.accuracy = correct.float().mean().item()
    rule_changes = (rules[:, 1:] != rules[:, :-1])
    pe = po = 0; sw_acc = []; st_acc = []; rec = []
    for si in range(n_seq):
        sp = torch.where(rule_changes[si])[0] + 1
        prev = 0
        for t in sp:
            t = t.item()
            if t - prev >= 10:
                a = correct[si, prev+5:t-5].float().mean().item()
                if not np.isnan(a):
                    st_acc.append(a)
            prev = t
        for t in sp:
            t = t.item()
            if t >= slen - 10:
                continue
            we = min(t + 10, slen)
            for k in range(t, we):
                if not correct[si, k]:
                    po += 1
                    if k > 0 and predictions[si, k] == predictions[si, k-1]:
                        pe += 1
            pa = correct[si, t:min(t+5, slen)].float().mean().item()
            if not np.isnan(pa):
                sw_acc.append(pa)
            for rt in range(t, min(slen - 5, t + 50)):
                if correct[si, rt:rt+5].float().mean().item() >= 0.8:
                    rec.append(rt - t); break
            else:
                rec.append(50)
    if po > 0:
        metrics.perseverative_error_rate = pe / po
    if sw_acc and st_acc:
        metrics.switch_cost = np.mean(st_acc) - np.mean(sw_acc)
        metrics.flexibility_index = np.mean(sw_acc) / (np.mean(st_acc) + 1e-8)
    if rec:
        metrics.trials_to_recover = np.mean(rec)
    if st_acc:
        metrics.rule_inference_accuracy = np.mean(st_acc)
    metrics.repetition_rate = (predictions[:, 1:] == predictions[:, :-1]).float().mean().item()
    oc = torch.bincount(predictions.flatten(), minlength=CONFIG['output_dim']).float()
    op = oc / oc.sum(); ent = -(op * torch.log(op + 1e-8)).sum().item()
    metrics.repetition_entropy = ent; metrics.output_diversity = ent / np.log(CONFIG['output_dim'])
    metrics.sparsity = model.get_sparsity(); metrics.stress_level = model.stress_level
    metrics.glutamate_noise = model.glutamate_noise
    metrics.inhibition_strength = model.inhibition_strength; metrics.use_tanh = model.use_tanh
    return metrics


# =============================================================================
# TRAINING
# =============================================================================
def train_epoch(model, train_loader, optimizer, criterion, device, pruning_manager=None):
    model.train(); tl = 0.0; nb = 0
    for data, labels, _ in train_loader:
        data, labels = data.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, _ = model(data)
        loss = criterion(logits.view(-1, CONFIG['output_dim']), labels.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if pruning_manager is not None:
            pruning_manager.apply_masks()
        tl += loss.item(); nb += 1
    return tl / nb


def train(model, train_loader=None, test_loader=None, epochs=None, lr=None,
          pruning_manager=None, verbose=True, eval_interval=10):
    if train_loader is None or test_loader is None:
        train_loader, test_loader, _ = create_rule_switch_dataloaders()
    epochs = epochs or CONFIG['baseline_epochs']
    lr = lr or CONFIG['baseline_lr']
    device = next(model.parameters()).device
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {'loss': [], 'accuracy': [], 'perseveration': []}
    for epoch in range(epochs):
        loss = train_epoch(model, train_loader, optimizer, criterion, device, pruning_manager)
        history['loss'].append(loss)
        if (epoch + 1) % eval_interval == 0 or epoch == epochs - 1:
            m = compute_ocd_metrics(model, test_loader, device)
            history['accuracy'].append(m.accuracy)
            history['perseveration'].append(m.perseverative_error_rate)
            if verbose:
                print(f"    Epoch {epoch+1:3d}/{epochs}: Loss={loss:.4f}, "
                      f"Acc={m.accuracy:.4f}, Persev={m.perseverative_error_rate:.4f}")
    return history


def train_with_stress_schedule(model, train_loader, test_loader, epochs, lr,
                               initial_stress, final_stress=0.0,
                               pruning_manager=None, verbose=False):
    device = next(model.parameters()).device
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(); losses = []
    for epoch in range(epochs):
        model.reduce_stress_gradually(epoch, epochs, initial_stress, final_stress)
        model.train(); el = 0.0; nb = 0
        for data, labels, _ in train_loader:
            data, labels = data.to(device), labels.to(device)
            optimizer.zero_grad()
            logits, _ = model(data)
            loss = criterion(logits.view(-1, CONFIG['output_dim']), labels.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            if pruning_manager:
                pruning_manager.apply_masks()
            el += loss.item(); nb += 1
        losses.append(el / nb)
    model.set_stress(final_stress)
    return losses


# =============================================================================
# THREE TREATMENT PROTOCOLS
# =============================================================================
def ketamine_treatment_ocd(model, pruning_mgr, train_loader, test_loader,
                           regrow_fraction=None, consolidation_epochs=None, verbose=True):
    if regrow_fraction is None:
        regrow_fraction = CONFIG['comparison_ketamine_regrow']
    if consolidation_epochs is None:
        consolidation_epochs = CONFIG['comparison_ketamine_epochs']
    if verbose:
        print(f"      [KETAMINE] regrow_fraction={regrow_fraction:.2f}, "
              f"consolidation={consolidation_epochs} epochs")
    rs = pruning_mgr.gradient_guided_regrow(train_loader, regrow_fraction=regrow_fraction)
    train(model, train_loader, test_loader, epochs=consolidation_epochs,
          lr=CONFIG['finetune_lr'], pruning_manager=pruning_mgr, verbose=False)
    return {'treatment': 'ketamine', 'mechanism': 'structural',
            'regrown': rs['connections_regrown'], 'regrow_fraction': regrow_fraction,
            'consolidation_epochs': consolidation_epochs,
            'final_sparsity': pruning_mgr.get_sparsity()}


def ssri_treatment_ocd(model, pruning_mgr, train_loader, test_loader,
                       epochs=None, lr=None, initial_stress=None, verbose=True):
    if epochs is None:
        epochs = CONFIG['comparison_ssri_epochs']
    if lr is None:
        lr = CONFIG['comparison_ssri_lr']
    if initial_stress is None:
        initial_stress = CONFIG['comparison_ssri_initial_stress']
    if verbose:
        print(f"      [SSRI] epochs={epochs}, lr={lr:.0e}, initial_stress={initial_stress:.2f}")
    isp = pruning_mgr.get_sparsity()
    train_with_stress_schedule(model, train_loader, test_loader, epochs=epochs, lr=lr,
                               initial_stress=initial_stress, final_stress=0.0,
                               pruning_manager=pruning_mgr, verbose=False)
    return {'treatment': 'ssri', 'mechanism': 'functional', 'epochs': epochs, 'lr': lr,
            'initial_stress': initial_stress, 'final_stress': model.stress_level,
            'initial_sparsity': isp, 'final_sparsity': pruning_mgr.get_sparsity()}


def neurosteroid_treatment_ocd(model, pruning_mgr, train_loader, test_loader,
                               strength=None, use_tanh=None, consolidation_epochs=None,
                               verbose=True):
    if strength is None:
        strength = CONFIG['comparison_neurosteroid_strength']
    if use_tanh is None:
        use_tanh = CONFIG['comparison_neurosteroid_use_tanh']
    if consolidation_epochs is None:
        consolidation_epochs = CONFIG['comparison_neurosteroid_epochs']
    if verbose:
        print(f"      [NEUROSTEROID] strength={strength:.2f}, use_tanh={use_tanh}, "
              f"consolidation={consolidation_epochs} epochs")
    isp = pruning_mgr.get_sparsity()
    model.set_inhibition(strength, use_tanh)
    train(model, train_loader, test_loader, epochs=consolidation_epochs,
          lr=CONFIG['finetune_lr'], pruning_manager=pruning_mgr, verbose=False)
    return {'treatment': 'neurosteroid', 'mechanism': 'functional_medication_dependent',
            'strength': strength, 'use_tanh': use_tanh,
            'consolidation_epochs': consolidation_epochs,
            'initial_sparsity': isp, 'final_sparsity': pruning_mgr.get_sparsity()}


# =============================================================================
# ISO-DOSE SWEEPS + TRUE (INTERPOLATED) MATCHING
# =============================================================================
def _clone_pruned(base_state, base_masks, device):
    model = CSTCNetwork().to(device)
    model.load_state_dict(copy.deepcopy(base_state))
    mgr = CSTCPruningManager(model)
    mgr.masks = copy.deepcopy(base_masks); mgr.apply_masks()
    return model, mgr


def run_ketamine_sweep(base_state, base_masks, train_loader, test_loader, device,
                       regrow_fractions=None):
    if regrow_fractions is None:
        regrow_fractions = CONFIG['ketamine_regrow_sweep']
    results = []
    for rf in regrow_fractions:
        model, mgr = _clone_pruned(base_state, base_masks, device)
        pre = {n: p.data.clone() for n, p in model.named_parameters() if 'weight' in n}
        ti = ketamine_treatment_ocd(model, mgr, train_loader, test_loader,
                                    regrow_fraction=rf, verbose=False)
        dose = compute_all_dose_metrics(pre, model)
        acute = compute_ocd_metrics(model, test_loader, device)
        prp = acute.perseverative_error_rate
        mgr.secondary_prune(fraction=CONFIG['relapse_prune_fraction'])
        rel = compute_ocd_metrics(model, test_loader, device)
        results.append({'treatment': 'ketamine', 'param_name': 'regrow_fraction',
                        'param_value': rf, 'dose': dose,
                        'acute_persev': acute.perseverative_error_rate,
                        'acute_flex': acute.flexibility_index, 'acute_accuracy': acute.accuracy,
                        'relapse_persev': rel.perseverative_error_rate,
                        'relapse_delta': rel.perseverative_error_rate - prp,
                        'treatment_info': ti})
    return results


def run_ssri_sweep(base_state, base_masks, train_loader, test_loader, device,
                   epochs_list=None, lr_list=None):
    if epochs_list is None:
        epochs_list = CONFIG['ssri_epochs_sweep']
    if lr_list is None:
        lr_list = [CONFIG['comparison_ssri_lr']]
    results = []
    for epochs in epochs_list:
        for lr in lr_list:
            model, mgr = _clone_pruned(base_state, base_masks, device)
            pre = {n: p.data.clone() for n, p in model.named_parameters() if 'weight' in n}
            ti = ssri_treatment_ocd(model, mgr, train_loader, test_loader,
                                    epochs=epochs, lr=lr, verbose=False)
            dose = compute_all_dose_metrics(pre, model)
            acute = compute_ocd_metrics(model, test_loader, device)
            prp = acute.perseverative_error_rate
            mgr.secondary_prune(fraction=CONFIG['relapse_prune_fraction'])
            rel = compute_ocd_metrics(model, test_loader, device)
            results.append({'treatment': 'ssri', 'param_name': 'epochs', 'param_value': epochs,
                            'lr': lr, 'dose': dose,
                            'acute_persev': acute.perseverative_error_rate,
                            'acute_flex': acute.flexibility_index, 'acute_accuracy': acute.accuracy,
                            'relapse_persev': rel.perseverative_error_rate,
                            'relapse_delta': rel.perseverative_error_rate - prp,
                            'treatment_info': ti})
    return results


def run_neurosteroid_sweep(base_state, base_masks, train_loader, test_loader, device,
                           strength_list=None):
    if strength_list is None:
        strength_list = CONFIG['neurosteroid_strength_sweep']
    results = []
    for st in strength_list:
        model, mgr = _clone_pruned(base_state, base_masks, device)
        pre = {n: p.data.clone() for n, p in model.named_parameters() if 'weight' in n}
        ti = neurosteroid_treatment_ocd(model, mgr, train_loader, test_loader,
                                        strength=st, verbose=False)
        dose = compute_all_dose_metrics(pre, model)
        acute = compute_ocd_metrics(model, test_loader, device)
        model.set_inhibition(1.0, False)
        off = compute_ocd_metrics(model, test_loader, device)
        model.set_inhibition(st, CONFIG['comparison_neurosteroid_use_tanh'])
        prp = acute.perseverative_error_rate
        mgr.secondary_prune(fraction=CONFIG['relapse_prune_fraction'])
        rel = compute_ocd_metrics(model, test_loader, device)
        results.append({'treatment': 'neurosteroid', 'param_name': 'strength', 'param_value': st,
                        'dose': dose, 'acute_persev': acute.perseverative_error_rate,
                        'acute_flex': acute.flexibility_index, 'acute_accuracy': acute.accuracy,
                        'off_med_persev': off.perseverative_error_rate,
                        'off_med_reversal': off.perseverative_error_rate - acute.perseverative_error_rate,
                        'relapse_persev': rel.perseverative_error_rate,
                        'relapse_delta': rel.perseverative_error_rate - prp,
                        'treatment_info': ti})
    return results


def find_iso_dose_params(sweep_results, target_dose, tolerance=None):
    """Nearest-neighbour match (kept for compatibility)."""
    if tolerance is None:
        tolerance = CONFIG['iso_dose_tolerance']
    best, bd = None, float('inf')
    for r in sweep_results:
        d = abs(r['dose'].l1_norm - target_dose)
        if d < bd:
            bd, best = d, r
    return best


def find_best_iso_dose_match(sweep_results, target_l1=0.005, tolerance=None):
    """
    TRUE iso-dose matching: linear interpolation in L1-dose space between the two
    bracketing sweep points. Returns interpolated param + outcomes AND the nearest
    actually-run config, with explicit residual mismatch for transparency.
    """
    if tolerance is None:
        tolerance = CONFIG['iso_dose_tolerance']
    rs = sorted(sweep_results, key=lambda r: r['dose'].l1_norm)
    doses = [r['dose'].l1_norm for r in rs]
    nearest = min(rs, key=lambda r: abs(r['dose'].l1_norm - target_l1))
    residual = abs(nearest['dose'].l1_norm - target_l1)
    interp = None
    for i in range(len(rs) - 1):
        d0, d1 = doses[i], doses[i + 1]
        if d0 <= target_l1 <= d1 and d1 > d0:
            w = (target_l1 - d0) / (d1 - d0)
            def lin(a, b):
                return a + w * (b - a)
            interp = {
                'param_name': rs[i]['param_name'],
                'param_value_interp': lin(float(rs[i]['param_value']), float(rs[i+1]['param_value'])),
                'acute_persev_interp': lin(rs[i]['acute_persev'], rs[i+1]['acute_persev']),
                'relapse_delta_interp': lin(rs[i]['relapse_delta'], rs[i+1]['relapse_delta']),
                'bracket': (rs[i]['param_value'], rs[i+1]['param_value'])}
            break
    return {'target_l1': target_l1, 'nearest': nearest, 'residual_mismatch': residual,
            'within_tolerance': residual <= tolerance, 'interpolated': interp}


# =============================================================================
# ISO-DOSE COMPARISON EXPERIMENT (single seed)
# =============================================================================
def run_iso_dose_comparison_experiment(device, seed=None, verbose=True):
    if seed is None:
        seed = CONFIG['seed']
    set_seed(seed)
    if verbose:
        print_section_header("ISO-DOSE FAIR COMPARISON EXPERIMENT", char="#")
        print(f"\n  Seed: {seed} | Device: {device}")
        print(f"  Prune mode: {CONFIG['prune_mode']} | baseline sparsity: {CONFIG['ocd_prune_sparsity']}")

    train_loader, test_loader, _ = create_rule_switch_dataloaders()

    base_model = CSTCNetwork().to(device)
    if verbose:
        print_subsection_header("PHASE 1: Shared Pruned Baseline (biologically motivated)")
        print("  Training healthy baseline...")
    train(base_model, train_loader, test_loader, verbose=False)
    base_mgr = CSTCPruningManager(base_model)
    ps = base_mgr.prune_baseline(train_loader, CONFIG['ocd_prune_sparsity'])
    if verbose:
        print(f"  Achieved sparsity: {ps['achieved_sparsity']*100:.1f}%")

    base_state = copy.deepcopy(base_model.state_dict())
    base_masks = copy.deepcopy(base_mgr.masks)
    untreated = compute_ocd_metrics(base_model, test_loader, device)
    if verbose:
        print(f"\n  UNTREATED: Persev={untreated.perseverative_error_rate:.4f} "
              f"Flex={untreated.flexibility_index:.4f} Acc={untreated.accuracy:.4f}")

    results = {'untreated': {'persev': untreated.perseverative_error_rate,
                             'flex': untreated.flexibility_index,
                             'accuracy': untreated.accuracy, 'sparsity': untreated.sparsity},
               'sweeps': {}, 'iso_dose_comparisons': {}}

    if verbose:
        print_subsection_header("PHASE 2: Parameter Sweeps")
    ket = run_ketamine_sweep(base_state, base_masks, train_loader, test_loader, device)
    ssri = run_ssri_sweep(base_state, base_masks, train_loader, test_loader, device)
    neuro = run_neurosteroid_sweep(base_state, base_masks, train_loader, test_loader, device)
    results['sweeps'] = {'ketamine': ket, 'ssri': ssri, 'neurosteroid': neuro}

    # Composite-dose scaling ranges (computational convenience)
    all_l1 = [r['dose'].l1_norm for r in ket + ssri + neuro]
    all_l2 = [r['dose'].l2_norm for r in ket + ssri + neuro]
    all_to = [r['dose'].synaptic_turnover for r in ket + ssri + neuro]
    all_sp = [r['dose'].sparsity_change for r in ket + ssri + neuro]
    scales = {'l1': (min(all_l1), max(all_l1)), 'l2': (min(all_l2), max(all_l2)),
              'turnover': (min(all_to), max(all_to)), 'sparsity': (min(all_sp), max(all_sp))}
    results['composite_scales'] = scales

    if verbose:
        print_subsection_header("PHASE 3: Dose-Response (multi-metric)")
        for label, rs in [('KETAMINE', ket), ('SSRI', ssri), ('NEUROSTEROID', neuro)]:
            print(f"\n  {label}:")
            print(f"  {'param':>10} {'L1':>10} {'Turnover':>10} {'Composite':>10} "
                  f"{'AcutePrsv':>11} {'Relapse Δ':>11}")
            for r in rs:
                comp = composite_dose(r['dose'], scales)
                print(f"  {r['param_value']:>10} {r['dose'].l1_norm:>10.6f} "
                      f"{r['dose'].synaptic_turnover:>10.4f} {comp:>10.4f} "
                      f"{r['acute_persev']:>11.4f} {r['relapse_delta']:>+11.4f}")

    if verbose:
        print_subsection_header("PHASE 4: TRUE Iso-Dose Matching (interpolated)")
    dose_range = (min(all_l1), max(all_l1))
    targets = [d for d in CONFIG['iso_dose_target_doses'] if dose_range[0] <= d <= dose_range[1]]
    if not targets:
        targets = [float(np.percentile(all_l1, p)) for p in (25, 50, 75)]
    for td in targets:
        comp = {}
        for nm, rs in [('ketamine', ket), ('ssri', ssri), ('neurosteroid', neuro)]:
            comp[nm] = find_best_iso_dose_match(rs, td)
        results['iso_dose_comparisons'][td] = comp
        if verbose:
            print(f"\n  TARGET L1 DOSE: {td:.6f}")
            print(f"  {'Treatment':<14}{'Nearest param':<22}{'Actual L1':>11}"
                  f"{'Residual':>11}{'InTol':>7}{'AcutePrsv':>11}")
            for nm in ['ketamine', 'ssri', 'neurosteroid']:
                m = comp[nm]['nearest']; res = comp[nm]['residual_mismatch']
                tol = comp[nm]['within_tolerance']
                ps_str = f"{m['param_name']}={m['param_value']}"
                print(f"  {nm.capitalize():<14}{ps_str:<22}{m['dose'].l1_norm:>11.6f}"
                      f"{res:>11.6f}{str(tol):>7}{m['acute_persev']:>11.4f}")

    # Efficiency + summary
    def stats(rs):
        doses = [r['dose'].l1_norm for r in rs]
        reds = [untreated.perseverative_error_rate - r['acute_persev'] for r in rs]
        eff = [rr / (d + 1e-8) for rr, d in zip(reds, doses)]
        return {'dose_range': (min(doses), max(doses)),
                'best_acute_persev': min(r['acute_persev'] for r in rs),
                'best_relapse_delta': min(r['relapse_delta'] for r in rs),
                'max_efficiency': max(eff), 'mean_efficiency': float(np.mean(eff))}
    results['summary'] = {'ketamine': stats(ket), 'ssri': stats(ssri),
                          'neurosteroid': stats(neuro)}
    if verbose:
        print_subsection_header("PHASE 5: Summary (single seed)")
        print(f"  {'Metric':<22}{'Ketamine':>14}{'SSRI':>14}{'Neurosteroid':>14}")
        for k, lbl in [('best_acute_persev', 'Best AcutePrsv'),
                       ('best_relapse_delta', 'Best Relapse Δ'),
                       ('max_efficiency', 'Max Efficiency'),
                       ('mean_efficiency', 'Mean Efficiency')]:
            print(f"  {lbl:<22}{results['summary']['ketamine'][k]:>14.4f}"
                  f"{results['summary']['ssri'][k]:>14.4f}"
                  f"{results['summary']['neurosteroid'][k]:>14.4f}")
    return results


# =============================================================================
# MULTI-SEED STATISTICS (Major #4: mean +/- SD, n>=5)
# =============================================================================
def run_with_statistics(device, n_seeds=None, verbose=True):
    n_seeds = n_seeds or CONFIG['n_seeds']
    print_section_header(f"MULTI-SEED STATISTICS (n={n_seeds})", char="#")
    collected = {'ketamine': {'best_acute_persev': [], 'mean_efficiency': [],
                              'best_relapse_delta': []},
                 'ssri': {'best_acute_persev': [], 'mean_efficiency': [],
                          'best_relapse_delta': []},
                 'neurosteroid': {'best_acute_persev': [], 'mean_efficiency': [],
                                  'best_relapse_delta': []}}
    untreated_persev = []
    for s in range(n_seeds):
        if verbose:
            print(f"\n  --- Seed {s} ---")
        res = run_iso_dose_comparison_experiment(device, seed=s, verbose=False)
        untreated_persev.append(res['untreated']['persev'])
        for t in collected:
            for k in collected[t]:
                collected[t][k].append(res['summary'][t][k])
    print(f"\n  Untreated perseveration: {np.mean(untreated_persev):.4f} "
          f"± {np.std(untreated_persev):.4f}")
    print(f"\n  {'Metric':<22}{'Ketamine (μ±σ)':>22}{'SSRI (μ±σ)':>22}"
          f"{'Neurosteroid (μ±σ)':>24}")
    print("  " + "-" * 88)
    for k, lbl in [('best_acute_persev', 'Best AcutePrsv'),
                   ('mean_efficiency', 'Mean Efficiency'),
                   ('best_relapse_delta', 'Best Relapse Δ')]:
        row = f"  {lbl:<22}"
        for t in ['ketamine', 'ssri', 'neurosteroid']:
            v = collected[t][k]
            row += f"{np.mean(v):>10.4f}±{np.std(v):<10.4f}".rjust(22 if t != 'neurosteroid' else 24)
        print(row)
    return {'untreated_persev': untreated_persev, 'collected': collected}


# =============================================================================
# SENSITIVITY / ABLATION BATTERY (Major #4)
# =============================================================================
def run_sensitivity_battery(device, verbose=True):
    print_section_header("SENSITIVITY & ABLATION BATTERY", char="#")
    out = {}

    # (A) Pruning sparsity sensitivity
    print_subsection_header("(A) Pruning sparsity sensitivity [0.40-0.70]")
    sp_rows = []
    orig_sp = CONFIG['ocd_prune_sparsity']
    for sp in CONFIG['prune_sensitivity_range']:
        CONFIG['ocd_prune_sparsity'] = sp
        res = run_iso_dose_comparison_experiment(device, seed=CONFIG['seed'], verbose=False)
        sp_rows.append((sp, res['untreated']['persev'],
                        res['summary']['ketamine']['best_acute_persev'],
                        res['summary']['ssri']['best_acute_persev'],
                        res['summary']['neurosteroid']['best_acute_persev']))
        print(f"   sparsity={sp:.2f}: untreated={sp_rows[-1][1]:.4f} | "
              f"ket={sp_rows[-1][2]:.4f} ssri={sp_rows[-1][3]:.4f} neuro={sp_rows[-1][4]:.4f}")
    CONFIG['ocd_prune_sparsity'] = orig_sp
    out['sparsity'] = sp_rows

    # (B) Architecture ablation: GRU vs LSTM ; modular on/off ; recurrence bias on/off
    print_subsection_header("(B) Architecture / mechanism ablations")
    ablations = [
        ('GRU+modular',  {'rnn_type': 'gru',  'use_modular': True,  'recurrence_bias': 1.2}),
        ('GRU-no_modular', {'rnn_type': 'gru', 'use_modular': False, 'recurrence_bias': 1.2}),
        ('LSTM+modular', {'rnn_type': 'lstm', 'use_modular': True,  'recurrence_bias': 1.2}),
        ('GRU+modular-no_recbias', {'rnn_type': 'gru', 'use_modular': True, 'recurrence_bias': 1.0}),
    ]
    saved = {k: CONFIG[k] for k in ('rnn_type', 'use_modular', 'recurrence_bias')}
    ab_rows = []
    for label, cfg in ablations:
        CONFIG.update(cfg)
        res = run_iso_dose_comparison_experiment(device, seed=CONFIG['seed'], verbose=False)
        ab_rows.append((label, res['untreated']['persev'],
                        res['summary']['ketamine']['best_acute_persev'],
                        res['summary']['ketamine']['mean_efficiency']))
        print(f"   {label:<26} untreated={ab_rows[-1][1]:.4f} | "
              f"ket_best={ab_rows[-1][2]:.4f} ket_eff={ab_rows[-1][3]:.2f}")
    CONFIG.update(saved)
    out['architecture'] = ab_rows

    # (C) Task difficulty: switch interval / noise
    print_subsection_header("(C) Task difficulty (switch_interval, input noise)")
    diff_rows = []
    for si, noise in [(30, 0.8), (50, 0.8), (50, 1.2), (80, 0.8)]:
        tr, te, _ = create_rule_switch_dataloaders(switch_interval=si, noise=noise)
        bm = CSTCNetwork().to(device)
        train(bm, tr, te, verbose=False)
        mgr = CSTCPruningManager(bm)
        mgr.prune_baseline(tr, CONFIG['ocd_prune_sparsity'])
        m = compute_ocd_metrics(bm, te, device)
        diff_rows.append((si, noise, m.perseverative_error_rate, m.flexibility_index))
        print(f"   switch={si} noise={noise}: persev={m.perseverative_error_rate:.4f} "
              f"flex={m.flexibility_index:.4f}")
    out['task'] = diff_rows
    return out


# =============================================================================
# MULTI-MECHANISM COMPARISON (single seed, with cumulative relapse)
# =============================================================================
def run_multi_mechanism_ocd_experiment(device, seed=None, verbose=True):
    if seed is None:
        seed = CONFIG['seed']
    set_seed(seed)
    print_section_header("MULTI-MECHANISM ANTIDEPRESSANT COMPARISON", char="#")
    train_loader, test_loader, _ = create_rule_switch_dataloaders()

    print_subsection_header("PHASE 1: Shared Pruned Baseline")
    base_model = CSTCNetwork().to(device)
    print("  Training healthy baseline...")
    train(base_model, train_loader, test_loader, verbose=False)
    base_mgr = CSTCPruningManager(base_model)
    ps = base_mgr.prune_baseline(train_loader, CONFIG['ocd_prune_sparsity'])
    print(f"  Prune mode={CONFIG['prune_mode']} | achieved sparsity: {ps['achieved_sparsity']*100:.1f}%")
    base_state = copy.deepcopy(base_model.state_dict())
    base_masks = copy.deepcopy(base_mgr.masks)

    untreated = compute_ocd_metrics(base_model, test_loader, device)
    results = {'untreated': {'sparsity': untreated.sparsity, 'accuracy': untreated.accuracy,
                             'persev': untreated.perseverative_error_rate,
                             'flex_index': untreated.flexibility_index}}
    print(f"\n  UNTREATED: Acc={untreated.accuracy:.4f} "
          f"Persev={untreated.perseverative_error_rate:.4f} "
          f"Flex={untreated.flexibility_index:.4f}")

    for treatment in ['ketamine', 'ssri', 'neurosteroid']:
        print_subsection_header(f"PHASE 3: {treatment.upper()} Treatment")
        model, mgr = _clone_pruned(base_state, base_masks, device)
        pre = {n: p.data.clone() for n, p in model.named_parameters() if 'weight' in n}
        if treatment == 'ketamine':
            ti = ketamine_treatment_ocd(model, mgr, train_loader, test_loader, verbose=verbose)
        elif treatment == 'ssri':
            ti = ssri_treatment_ocd(model, mgr, train_loader, test_loader, verbose=verbose)
        else:
            ti = neurosteroid_treatment_ocd(model, mgr, train_loader, test_loader, verbose=verbose)

        dm = compute_all_dose_metrics(pre, model)
        acute = compute_ocd_metrics(model, test_loader, device)

        off = None
        if treatment == 'neurosteroid':
            model.set_inhibition(1.0, False)
            off = compute_ocd_metrics(model, test_loader, device)
            model.set_inhibition(CONFIG['comparison_neurosteroid_strength'],
                                 CONFIG['comparison_neurosteroid_use_tanh'])

        # CONCEPTUAL relapse: cumulative or single
        if CONFIG['relapse_mode'] == 'cumulative':
            traj = mgr.cumulative_relapse(model, test_loader, device)
            relapse = compute_ocd_metrics(model, test_loader, device)
        else:
            mgr.secondary_prune(fraction=CONFIG['relapse_prune_fraction'])
            relapse = compute_ocd_metrics(model, test_loader, device)
            traj = None

        results[treatment] = {
            'dose_metrics': {'l1_norm': dm.l1_norm, 'l2_norm': dm.l2_norm,
                             'synaptic_turnover': dm.synaptic_turnover,
                             'sparsity_change': dm.sparsity_change},
            'acute_sparsity': acute.sparsity, 'acute_accuracy': acute.accuracy,
            'acute_persev': acute.perseverative_error_rate, 'acute_flex': acute.flexibility_index,
            'improvement_persev': untreated.perseverative_error_rate - acute.perseverative_error_rate,
            'relapse_persev': relapse.perseverative_error_rate,
            'relapse_delta_persev': relapse.perseverative_error_rate - acute.perseverative_error_rate,
            'off_med_persev': off.perseverative_error_rate if off else None,
            'relapse_trajectory': traj}

        eff = results[treatment]['improvement_persev'] / (dm.l1_norm + 1e-8)
        print(f"    DOSE: L1={dm.l1_norm:.6f} Turnover={dm.synaptic_turnover:.4f} "
              f"ΔSparsity={dm.sparsity_change:.4f}")
        print(f"    ACUTE: Acc={acute.accuracy:.4f} Persev={acute.perseverative_error_rate:.4f} "
              f"Flex={acute.flexibility_index:.4f} | Efficiency={eff:.2f}")
        if off:
            print(f"    OFF-MED reversal: {off.perseverative_error_rate - acute.perseverative_error_rate:+.4f}")
        print(f"    RELAPSE ({CONFIG['relapse_mode']}): Persev={relapse.perseverative_error_rate:.4f} "
              f"(Δ={results[treatment]['relapse_delta_persev']:+.4f})")

    print_section_header("COMPREHENSIVE COMPARISON", char="=")
    print(f"  {'Treatment':<14}{'L1 Dose':>11}{'Turnover':>11}{'AcutePrsv':>11}"
          f"{'Efficiency':>12}{'Relapse Δ':>11}")
    print("  " + "-" * 70)
    for t in ['ketamine', 'ssri', 'neurosteroid']:
        r = results[t]
        eff = r['improvement_persev'] / (r['dose_metrics']['l1_norm'] + 1e-8)
        print(f"  {t.capitalize():<14}{r['dose_metrics']['l1_norm']:>11.6f}"
              f"{r['dose_metrics']['synaptic_turnover']:>11.4f}{r['acute_persev']:>11.4f}"
              f"{eff:>12.2f}{r['relapse_delta_persev']:>+11.4f}")
    return results


# =============================================================================
# PLAIN-LANGUAGE MODELING ASSUMPTIONS & LIMITATIONS  (Moderate #6-8)
# =============================================================================
def print_limitations():
    print_section_header("MODELING ASSUMPTIONS & LIMITATIONS (PLAIN LANGUAGE)", char="#")
    print("""
  This is HYPOTHESIS-GENERATING in-silico exploration — NOT a validated
  mechanistic or clinical model. Please read every result through this lens.

  1. PRUNING is a COMPUTATIONAL PROXY. We use activity-dependent / weak-synapse
     ("complement-like") pruning at a 50-65% range as a stand-in for synaptic
     loss. It does NOT replicate microglia/complement-mediated, activity-
     dependent developmental refinement, and the sparsity value is not directly
     comparable to any human imaging measure.

  2. CSTC FIDELITY is LIMITED. The added striatal "habit" module and thalamic
     confidence gate are crude caricatures of cortico-striato-thalamo-cortical
     loops. There is no pathway-specific neurochemistry and only a schematic
     goal-directed vs habitual distinction.

  3. ISO-DOSE is a CONVENIENCE METRIC, NOT A PHARMACOLOGICAL DOSE. The L1/L2
     weight-change norms, turnover, and composite score quantify total NETWORK
     change. They are NOT receptor occupancy, plasma concentration, or clinical
     dose, and structural vs functional changes are not truly commensurable on
     one scalar. Residual dose mismatches are reported explicitly.

  4. TASK-PHENOTYPE MAPPING is NARROW. Rule-switching captures set-shifting
     rigidity only. The optional "ritual" variant is a toy compulsive-habit
     analogue; the model omits threat salience, affective dysregulation, and
     reward-based compulsive learning.

  5. RELAPSE is CONCEPTUAL. Cumulative, recurrence-biased re-pruning with rising
     stress is an illustrative device, not a biologically grounded relapse model.

  6. CLINICAL IMPLICATIONS ARE SPECULATIVE. Any treatment ordering (e.g.,
     structural > functional durability) holds ONLY within this abstraction and
     must NOT be read as evidence about ketamine, SSRIs, or neurosteroids in
     patients. No patient data calibrate these simulations.
""")


# =============================================================================
# MAIN
# =============================================================================
def main(run_full_statistics=True, run_sensitivity=True):
    print("\n" + "#" * 80)
    print("#" + "COMPUTATIONAL MODEL OF OCD-LIKE RIGIDITY".center(78) + "#")
    print("#" + "v4.0 BIOLOGICALLY-GROUNDED REVISION".center(78) + "#")
    print("#" * 80)
    print(f"\n  PyTorch: {torch.__version__} | Device: {DEVICE} | "
          f"CUDA: {torch.cuda.is_available()}")

    print_limitations()
    set_seed(CONFIG['seed'])

    print("\n  >>> Multi-Mechanism Comparison (single seed, cumulative relapse)")
    multi = run_multi_mechanism_ocd_experiment(DEVICE, verbose=True)

    print("\n  >>> Iso-Dose Comparison (single seed, true matching)")
    iso = run_iso_dose_comparison_experiment(DEVICE, verbose=True)

    stats = None
    if run_full_statistics:
        print("\n  >>> Multi-Seed Statistics (n>=5)")
        stats = run_with_statistics(DEVICE, n_seeds=CONFIG['n_seeds'], verbose=True)

    sens = None
    if run_sensitivity:
        print("\n  >>> Sensitivity & Ablation Battery")
        sens = run_sensitivity_battery(DEVICE, verbose=True)

    print_limitations()
    print("\n" + "#" * 80)
    print("#" + "EXPERIMENT COMPLETE (v4.0)".center(78) + "#")
    print("#" * 80)
    return {'multi_mechanism': multi, 'iso_dose': iso,
            'statistics': stats, 'sensitivity': sens}


if __name__ == "__main__":
    # NOTE: full statistics (n=5) + sensitivity battery is the heaviest path.
    # Should complete in well under ~2h on an A100. To iterate faster, call
    # main(run_full_statistics=False, run_sensitivity=False).
    results = main(run_full_statistics=True, run_sensitivity=True)


################################################################################
#                   COMPUTATIONAL MODEL OF OCD-LIKE RIGIDITY                   #
#                     v4.0 BIOLOGICALLY-GROUNDED REVISION                      #
################################################################################

  PyTorch: 2.11.0+cu128 | Device: cuda | CUDA: True

################################################################################
              MODELING ASSUMPTIONS & LIMITATIONS (PLAIN LANGUAGE)               
################################################################################

  This is HYPOTHESIS-GENERATING in-silico exploration — NOT a validated
  mechanistic or clinical model. Please read every result through this lens.

  1. PRUNING is a COMPUTATIONAL PROXY. We use activity-dependent / weak-synapse
     ("complement-like") pruning at a 50-65% range as a stand-in for synaptic
     loss. It does NOT replicate microglia/complement-mediated, activ

# The End